# HARE on Kaggle: System 0, enforcement and verification for Qwen3.5-4B

**Before running:** Settings → Accelerator **GPU T4 x2**, Internet **On** (phone-verified account).
Then **Save Version → Save & Run All (Commit)** and close the tab. Results are written after every episode and every arm, so an interrupted run keeps what finished.

Phases, in order:
1. **registered** (HARE-S0-1, unchanged): 4B plain vs 4B + advisory System 0 on the frozen 180 and the fresh 179.
2. **extension** (HARE-S0-2, registered separately): 4B plain / advisory / enforce / verify / full on 179 untouched episodes. With `RUN_35B`, also the 35B plain and full arms on the same T4s, for the non-inferiority test and the offline cascade.
3. **liquid** (optional probe): builds llama.cpp b12620 and runs Liquid d1-2B against the deterministic rules (accuracy, calibration, latency).

Synthetic data only. No credentials, no paid services. GPU time is roughly 2 h for phase 1, 2.5 h for phase 2 with the 4B only, plus about 1.5 h for the 35B and 30 min for the Liquid probe. Keep the notebook private, because the `*.jsonl` files hold full synthetic transcripts.

In [ ]:
PHASES = ["registered", "extension"]   # add "liquid" for the d1-2B probe
RUN_35B = False        # True: 35B plain + full on the same T4s (22 GB download)
SKIP_FRESH = False     # registered phase: frozen set only
HAVEN_COMMIT = "53387e74a0ad90786ac3ea16753c8ad99142e9df"
WORK = "/kaggle/working/hare"
MODELS = "/tmp/hare-models"
LIQUID = {"llama_tag": "b12620", "d1": {"repo": "LiquidAI/d1-2B-GGUF", "revision": "4a181094d1b6addfaa3a0956f78e7ab2aa406a0c", "file": "d1-2B-Q4_K_M.gguf", "sha256": "adca5789ad30ae6027e8e936749bb9f19673097492ab26fa1c5b69643ca50b4b"}}


In [ ]:
import subprocess, os, sys, shutil, json, hashlib, base64, io, tarfile, pathlib, time
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout or "")[-3000:], (r.stderr or "")[-3000:], flush=True)
    if check and r.returncode: raise SystemExit(f"failed: {cmd}")
    return r.stdout
def stream(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout: print(line, end="", flush=True)
    assert p.wait() == 0, f"failed: {cmd}"
sh("nvidia-smi --query-gpu=name,memory.total --format=csv"); sh("nvcc --version | tail -2")
sh("nproc; free -g | head -2; df -h /tmp /kaggle/working | tail -2")
os.makedirs(WORK, exist_ok=True); os.makedirs(MODELS, exist_ok=True)


In [ ]:
if not os.path.isdir("/tmp/haven/.git"):
    sh("git clone -q --filter=blob:none https://github.com/erpzz/haven /tmp/haven")
sh(f"git -C /tmp/haven checkout -q {HAVEN_COMMIT} && git -C /tmp/haven log -1 --format='%h %s'")
BLOB = "..."
raw = base64.b64decode(BLOB)
assert hashlib.sha256(raw).hexdigest() == "9ce26971ab51648412753fbbdc45f04dc244afbb84b7701dce4f8157e006994f", "embedded harness corrupted"
dest = pathlib.Path("/tmp/haven/labs/intelligence-bakeoff")
shutil.rmtree(dest / "system0", ignore_errors=True)
tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz").extractall(dest, filter="data")
S0 = dest / "system0"
sh(f"cd {dest} && python3 -m unittest discover -s system0/tests 2>&1 | tail -3")
sh(f"cd {dest} && python3 system0/bench_s0.py check")


In [ ]:
def build_llama(tag, prefix):
    engine = f"{prefix}/build/bin/llama-server"
    if not os.path.exists(engine):
        sh(f"git clone -q --depth 1 --branch {tag} https://github.com/ggml-org/llama.cpp {prefix}")
        t0 = time.time()
        # CUDA 13 images: CMake may not locate the driver library (CUDA::cuda_driver). Point it there.
        cands = sh("ls /usr/local/cuda/lib64/stubs/libcuda.so /usr/local/cuda/targets/x86_64-linux/lib/stubs/libcuda.so "
                   "/usr/lib/x86_64-linux-gnu/libcuda.so /usr/lib/x86_64-linux-gnu/libcuda.so.1 2>/dev/null", check=False).split()
        drv = f"-DCUDA_cuda_driver_LIBRARY={cands[0]}" if cands else ""
        flags = (f"-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 {drv} -DLLAMA_CURL=OFF -DLLAMA_USE_PREBUILT_UI=OFF "
                 "-DLLAMA_BUILD_UI=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF")
        cmake = "cmake"
        if subprocess.run(f"cd {prefix} && {cmake} -S . -B build {flags} > /tmp/cmake.log 2>&1", shell=True).returncode:
            print(open("/tmp/cmake.log").read()[-2500:])
            print("Retrying configure with a newer CMake from pip", flush=True)
            sh("pip install -q 'cmake>=3.31,<4.2'")
            import importlib, site
            importlib.invalidate_caches()
            cmake = sh("python3 -c 'import cmake,os;print(os.path.join(cmake.CMAKE_BIN_DIR,chr(99)+chr(109)+chr(97)+chr(107)+chr(101)))'").strip()
            sh(f"rm -rf {prefix}/build && cd {prefix} && {cmake} -S . -B build {flags} > /tmp/cmake.log 2>&1 || (tail -40 /tmp/cmake.log; false)")
        sh(f"cd {prefix} && {cmake} --build build --target llama-server -j4 > /tmp/build.log 2>&1 || (tail -60 /tmp/build.log; false)")
        print(f"{tag} build minutes: {(time.time()-t0)/60:.1f}")
    sh(f"{engine} --version 2>&1 | tail -2")
    return engine
ENGINE = build_llama("b11146", "/tmp/llama.cpp")   # pinned engine for every Qwen arm


In [ ]:
reg = json.loads((S0 / "PREREGISTRATION_S0.json").read_text())
def fetch(spec):
    path = f"{MODELS}/{spec['file']}"
    if not os.path.exists(path):
        url = f"https://huggingface.co/{spec['repo']}/resolve/{spec['revision']}/{spec['file']}"
        sh(f"curl -sSL --retry 5 -o {path}.part '{url}' && mv {path}.part {path}")
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(1 << 24), b""): h.update(c)
    assert h.hexdigest() == spec["sha256"], f"sha256 mismatch for {spec['file']}"
    return path
M4 = fetch(reg["settings"]["4b"])
M35 = None
if RUN_35B:
    free_gb = shutil.disk_usage(MODELS).free / 1e9
    M35 = fetch(reg["settings"]["35b"]) if free_gb >= 26 else print(f"Only {free_gb:.0f} GB free: skipping the 35B")
gpus = sh("nvidia-smi --query-gpu=name --format=csv,noheader").strip().splitlines()
HW = f"Kaggle {len(gpus)}x {gpus[0]}; 4B on GPU0; 35B layer-split across all GPUs (Slink used --cpu-moe)"


In [ ]:
if "registered" in PHASES:
    stream(f"cd {S0} && python3 run_arms.py --engine {ENGINE} --model-4b {M4} --out {WORK}/registered --hardware '{HW}'"
           + (f" --model-35b {M35}" if M35 else "") + (" --skip-fresh" if SKIP_FRESH else ""))
    print(json.dumps(json.loads(pathlib.Path(WORK, "registered", "results.json").read_text())["verdict"], indent=2))


In [ ]:
if "extension" in PHASES:
    stream(f"cd {S0} && python3 run_extension.py --engine {ENGINE} --model-4b {M4} --out {WORK}/extension --hardware '{HW}'"
           + (f" --model-35b {M35}" if M35 else ""))


In [ ]:
if "liquid" in PHASES:
    LENGINE = build_llama(LIQUID["llama_tag"], "/tmp/llama-new")
    D1 = fetch(LIQUID["d1"])
    srv = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=0 {LENGINE} -m {D1} --host 127.0.0.1 --port 18940 -c 8192 -b 8192 -ub 8192 -ngl 99 --no-webui",
                           shell=True, stdout=open(f"{WORK}/d1-server.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(200):
        time.sleep(3)
        if subprocess.run("curl -s http://127.0.0.1:18940/health | grep -q ok", shell=True).returncode == 0: break
    try:
        stream(f"cd {dest} && python3 system0/d1_probe.py --url http://127.0.0.1:18940 "
               f"--episodes frozen,system0/fresh-s0-20261121.jsonl --out {WORK}/d1-probe-t4.json")
    finally:
        srv.terminate()


In [ ]:
# Sanitized bundle to send back: summaries and results, no transcripts
sh(f"cd {WORK} && find . -name '*.summary.json' -o -name 'results*.json' -o -name 'd1-probe-*.json' | tar czf /kaggle/working/hare-results.tgz -T -")
print("Download /kaggle/working/hare-results.tgz from the Output tab and send it back.")
